# Evaluation and Output (Train/Test Only)
Use the trained model to predict on test set, map risk levels, and export outputs.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

test_df = pd.read_csv('../data/test_fe.csv')
model = joblib.load('../models/churn_model_train_test_only.joblib')

required_cols = {'id'}
missing_required = required_cols - set(test_df.columns)
if missing_required:
    raise ValueError(f'Missing required columns in test_fe.csv: {missing_required}')

X_test = test_df.drop(columns=['id'])
test_proba = model.predict_proba(X_test)[:, 1]
test_proba = np.clip(test_proba, 0.0, 1.0)

print('Predictions generated:', len(test_proba))
print('Probability summary:')
print(pd.Series(test_proba).describe())

# Optional: show selected model name if benchmark metadata exists
meta_path = Path('../models/model_benchmark.json')
if meta_path.exists():
    with meta_path.open('r', encoding='utf-8') as f:
        meta = json.load(f)
    print('Selected model from benchmark:', meta.get('selected_model'))

In [ ]:
def map_risk(p):
    if p < 0.33:
        return 'Low Risk'
    if p < 0.66:
        return 'Medium Risk'
    return 'High Risk'

risk_level = pd.Series(test_proba).apply(map_risk)
pred_label = (pd.Series(test_proba) >= 0.5).astype(int)

output_df = pd.DataFrame({
    'id': test_df['id'],
    'churn_probability': test_proba,
    'risk_level': risk_level,
    'Churn': pred_label,
})

output_df.head()

In [ ]:
print(output_df['risk_level'].value_counts())
print(output_df['risk_level'].value_counts(normalize=True))

In [ ]:
output_df.to_csv('../output/predictions_with_risk.csv', index=False)
output_df[['id', 'Churn']].to_csv('../output/submission.csv', index=False)
print('Saved ../output/predictions_with_risk.csv')
print('Saved ../output/submission.csv')